# DSPy — Technical Research Pipeline + ReAct + Ollama

**Objetivo:** construir una pipeline de investigación técnica que clasifique una pregunta, investigue mediante herramientas, redacte una respuesta y audite sus evidencias. **No requiere créditos de OpenAI.**

**Arquitectura:** `Predict → ReAct (tools) → ChainOfThought → Evidence Audit`.

> **Importante:** el notebook contiene llamadas opcionales a documentación pública (requieren Internet), pero funciona con un corpus local de documentación resumida. Los textos locales son **fichas didácticas**, no copias verificadas ni documentación oficial en tiempo real. La verificación de referencias **no garantiza** que una afirmación esté respaldada semánticamente.

## 0. Requisitos

1. Instalar [Ollama](https://ollama.com/) y dejarlo ejecutándose.
2. Descargar un modelo: `ollama pull qwen2.5:7b` (recomendado si tu equipo puede ejecutarlo). Si ya tenés `llama3.2`, podés usarlo cambiando `MODEL` más abajo; es posible que falle más al elegir herramientas.
3. Ejecutar las celdas en orden. Las llamadas a Ollama son locales, pero consumen recursos de tu computadora.

**Consejo:** empezá con `MAX_ITERS=3` y una sola pregunta. En ReAct, cada iteración puede generar varias llamadas al modelo.

In [ ]:
%pip install -U "dspy>=2.6" requests beautifulsoup4 pandas

## 1. Configurar Ollama y DSPy

Usamos `ollama_chat/` mediante LiteLLM, que DSPy utiliza para comunicarse con Ollama. No configuramos `OPENAI_API_KEY`.

In [ ]:
import os
import json
import re
import time
from collections import Counter
from pprint import pprint
from urllib.parse import urlparse

import dspy
import pandas as pd
import requests
from bs4 import BeautifulSoup

MODEL = os.getenv("OLLAMA_MODEL", "qwen2.5:7b")  # alternativa: "llama3.2"
OLLAMA_BASE = "http://localhost:11434"
MAX_ITERS = 3

try:
    r = requests.get(f"{OLLAMA_BASE}/api/tags", timeout=5)
    r.raise_for_status()
    installed = [m.get("name") for m in r.json().get("models", [])]
    print("Modelos instalados:", installed)
except requests.RequestException as exc:
    raise RuntimeError("Ollama no responde. Iniciá Ollama antes de seguir.") from exc

if not any(x == MODEL or x.startswith(MODEL + ":") for x in installed):
    print(f"AVISO: {MODEL!r} no figura instalado. Ejecutá: ollama pull {MODEL}")

lm = dspy.LM(
    f"ollama_chat/{MODEL}",
    api_base=OLLAMA_BASE,
    api_key="ollama",  # valor local ficticio; no es una clave de OpenAI
    temperature=0,
    max_tokens=1800,
    cache=False,
    num_retries=0,
)
dspy.configure(lm=lm)
print("DSPy:", getattr(dspy, "__version__", "desconocida"))
print("Modelo:", MODEL)

## 2. Corpus local de fichas técnicas

Cada documento tiene un identificador (`DOC-...`), tema, URL oficial de referencia y fragmentos breves. Las URLs indican dónde **contrastar** la ficha; **no** implican que el contenido haya sido descargado de esas páginas.

Las fichas permiten que el ejercicio sea reproducible sin depender de un buscador ni de una API externa.

In [ ]:
DOCS = {
    "DOC-FAPI-01": {
        "topic": "fastapi", "title": "FastAPI — Request validation",
        "url": "https://fastapi.tiangolo.com/tutorial/body/",
        "text": "FastAPI usa modelos Pydantic para validar el cuerpo de solicitudes. Si un campo obligatorio no está presente o tiene un valor inválido, puede responder HTTP 422 con detalles de validación.",
    },
    "DOC-FAPI-02": {
        "topic": "fastapi", "title": "FastAPI — JSON compatible encoder",
        "url": "https://fastapi.tiangolo.com/tutorial/encoder/",
        "text": "jsonable_encoder convierte objetos como datetime y modelos Pydantic en estructuras compatibles con JSON. Resulta útil cuando se necesita una estructura JSON compatible antes de enviarla a un componente que espera tipos JSON nativos.",
    },
    "DOC-PYD-01": {
        "topic": "pydantic", "title": "Pydantic — Fields",
        "url": "https://docs.pydantic.dev/latest/concepts/fields/",
        "text": "En Pydantic v2, un campo anotado con Optional[T] no tiene por sí solo un valor predeterminado. Para que no sea obligatorio se puede declarar un default explícito, por ejemplo field: str | None = None.",
    },
    "DOC-PYD-02": {
        "topic": "pydantic", "title": "Pydantic — Serialization",
        "url": "https://docs.pydantic.dev/latest/concepts/serialization/",
        "text": "model_dump() devuelve una representación Python; model_dump(mode='json') busca valores compatibles con JSON. model_dump_json() serializa a una cadena JSON.",
    },
    "DOC-SQLA-01": {
        "topic": "sqlalchemy", "title": "SQLAlchemy — AsyncIO",
        "url": "https://docs.sqlalchemy.org/en/20/orm/extensions/asyncio.html",
        "text": "Con SQLAlchemy asyncio, una carga implícita de atributos que realiza IO fuera de un contexto async esperado puede provocar MissingGreenlet. Se recomienda evitar IO implícito, considerar eager loading y controlar explícitamente la carga de relaciones.",
    },
    "DOC-SQLA-02": {
        "topic": "sqlalchemy", "title": "SQLAlchemy — Relationship loading",
        "url": "https://docs.sqlalchemy.org/en/20/orm/queryguide/relationships.html",
        "text": "selectinload() permite eager loading de relaciones mediante consultas adicionales controladas. Puede evitar accesos posteriores que disparen lazy loading inesperado.",
    },
    "DOC-PY-01": {
        "topic": "python", "title": "Python — json module",
        "url": "https://docs.python.org/3/library/json.html",
        "text": "json.dumps convierte objetos Python compatibles a texto JSON. Un datetime no es serializable directamente con el encoder JSON estándar sin una conversión o encoder personalizado.",
    },
}
print(f"Documentos disponibles: {len(DOCS)}")
pd.DataFrame([{"id": k, "topic": v["topic"], "title": v["title"]} for k, v in DOCS.items()])

## 3. Herramientas para ReAct

Son funciones Python ordinarias, con tipos y docstrings descriptivos. `search_docs` encuentra candidatos; `read_doc` recupera su contenido; `compare_docs` recupera dos fichas juntas. `fetch_official_page` es **opcional** y solo consulta URLs oficiales preaprobadas (sin URLs arbitrarias). Guardamos una auditoría de cada llamada.

In [ ]:
TOOL_AUDIT = []
RETRIEVED = {}


def record(tool: str, arguments: dict, result: dict) -> dict:
    TOOL_AUDIT.append({"tool": tool, "arguments": arguments, "result": result})
    return result


def search_docs(query: str) -> dict:
    """Busca fichas técnicas por palabras clave. Devuelve IDs, títulos y fragmentos breves. Usar antes de read_doc."""
    tokens = set(re.findall(r"[a-z0-9_]+", query.lower()))
    matches = []
    for doc_id, doc in DOCS.items():
        haystack = f'{doc["topic"]} {doc["title"]} {doc["text"]}'.lower()
        score = sum(token in haystack for token in tokens if len(token) > 2)
        if score:
            matches.append({"doc_id": doc_id, "title": doc["title"], "score": score, "preview": doc["text"][:140]})
    matches.sort(key=lambda item: item["score"], reverse=True)
    return record("search_docs", {"query": query}, {"matches": matches[:5]})


def read_doc(doc_id: str) -> dict:
    """Lee una ficha por ID exacto, por ejemplo DOC-SQLA-01. Devuelve su texto y URL de referencia."""
    doc = DOCS.get(doc_id)
    if not doc:
        return record("read_doc", {"doc_id": doc_id}, {"error": "Documento inexistente"})
    RETRIEVED[doc_id] = dict(doc)
    return record("read_doc", {"doc_id": doc_id}, {"doc_id": doc_id, **doc})


def compare_docs(first_id: str, second_id: str) -> dict:
    """Lee y compara dos fichas por sus IDs exactos. Útil para contrastar conceptos relacionados."""
    found = {}
    for doc_id in [first_id, second_id]:
        if doc_id in DOCS:
            RETRIEVED[doc_id] = dict(DOCS[doc_id])
            found[doc_id] = DOCS[doc_id]
    return record("compare_docs", {"first_id": first_id, "second_id": second_id}, {"documents": found})


TOOLS = [search_docs, read_doc, compare_docs]
print("Herramientas:", [t.__name__ for t in TOOLS])

### Opcional: consultar páginas oficiales en vivo

Esta herramienta **no se incluye por defecto**: depende de Internet, del HTML de cada sitio y puede devolver contenido incompleto. Solo acepta los IDs del corpus, nunca una URL proporcionada libremente por el modelo. Recorta el texto para evitar contextos excesivos.

In [ ]:
def fetch_official_page(doc_id: str) -> dict:
    """Consulta por HTTPS la URL oficial de una ficha conocida. Devuelve un extracto no verificado de la página actual."""
    doc = DOCS.get(doc_id)
    if not doc:
        return record("fetch_official_page", {"doc_id": doc_id}, {"error": "ID desconocido"})
    url = doc["url"]
    allowed = {"fastapi.tiangolo.com", "docs.pydantic.dev", "docs.sqlalchemy.org", "docs.python.org"}
    parsed = urlparse(url)
    if parsed.scheme != "https" or parsed.hostname not in allowed:
        return record("fetch_official_page", {"doc_id": doc_id}, {"error": "URL no permitida"})
    try:
        response = requests.get(url, timeout=8, headers={"User-Agent": "DSPy-Learning-Notebook/1.0"}, allow_redirects=False)
        response.raise_for_status()
        soup = BeautifulSoup(response.text[:250_000], "html.parser")
        for element in soup(["script", "style", "nav", "footer"]):
            element.decompose()
        body = soup.find("main") or soup.find("article") or soup.body or soup
        excerpt = " ".join(body.get_text(" ", strip=True).split())[:3500]
        result = {"doc_id": doc_id, "url": url, "live_excerpt": excerpt, "truncated": True}
        RETRIEVED[doc_id] = {**doc, "live_excerpt": excerpt}
    except requests.RequestException as exc:
        result = {"doc_id": doc_id, "error": type(exc).__name__, "detail": str(exc)[:180]}
    return record("fetch_official_page", {"doc_id": doc_id}, result)

# Para incluirla, ejecutar ANTES de crear la pipeline:
# TOOLS = [search_docs, read_doc, compare_docs, fetch_official_page]

## 4. Signatures: contratos entre etapas

Los campos de salida se tipan para facilitar la inspección. El investigador ReAct debe citar IDs concretos; el redactor debe conservar esos IDs en la respuesta.

In [ ]:
from typing import Literal

class ClassifyQuestion(dspy.Signature):
    """Clasifica la consulta técnica y propone términos para investigar. No respondas todavía."""
    question: str = dspy.InputField()
    category: Literal["python", "fastapi", "pydantic", "sqlalchemy", "other"] = dspy.OutputField()
    search_hint: str = dspy.OutputField(desc="Términos técnicos concretos para buscar")

class ResearchQuestion(dspy.Signature):
    """Investiga usando las herramientas disponibles. Cita únicamente IDs de documentos realmente leídos. Si falta evidencia, decláralo."""
    question: str = dspy.InputField()
    category: str = dspy.InputField()
    search_hint: str = dspy.InputField()
    findings: str = dspy.OutputField(desc="Hallazgos breves con IDs de las fuentes consultadas")
    source_ids: list[str] = dspy.OutputField(desc="IDs de fichas realmente leídas")

class WriteAnswer(dspy.Signature):
    """Redacta una respuesta técnica en español, usando solo los hallazgos proporcionados. Incluye IDs [DOC-...] junto a afirmaciones verificables; si faltan datos, dilo."""
    question: str = dspy.InputField()
    findings: str = dspy.InputField()
    source_ids: list[str] = dspy.InputField()
    answer: str = dspy.OutputField()
    caveats: list[str] = dspy.OutputField(desc="Limitaciones o dudas de la respuesta")

## 5. Auditoría de evidencias: dos niveles

**Nivel A — trazabilidad determinista:** comprueba que cada ID citado existe y fue recuperado por una herramienta. **Nivel B — apoyo semántico:** un segundo modelo puede juzgar si la evidencia respalda la afirmación, pero su evaluación es falible; no equivale a verificación factual independiente.

In [ ]:
CITATION_PATTERN = re.compile(r"DOC-[A-Z]+-\d+")


def audit_references(answer: str, retrieved: dict) -> dict:
    """Comprueba existencia y recuperación; NO comprueba apoyo semántico."""
    cited = sorted(set(CITATION_PATTERN.findall(answer)))
    known = set(DOCS)
    actually_read = set(retrieved)
    return {
        "cited_ids": cited,
        "valid_and_retrieved": sorted(set(cited) & actually_read),
        "known_but_not_retrieved": sorted((set(cited) & known) - actually_read),
        "unknown_ids": sorted(set(cited) - known),
        "retrieved_but_not_cited": sorted(actually_read - set(cited)),
        "has_traceable_citation": bool(set(cited) & actually_read),
    }

class CheckEvidence(dspy.Signature):
    """Evalúa si las afirmaciones de la respuesta están apoyadas por las fichas proporcionadas. Sé conservador: no inventes evidencia y marca dudas. Esta evaluación es orientativa."""
    answer: str = dspy.InputField()
    evidence: str = dspy.InputField()
    supported: bool = dspy.OutputField()
    unsupported_claims: list[str] = dspy.OutputField()
    explanation: str = dspy.OutputField()

## 6. Pipeline completa: `dspy.Module` y `forward()`

`forward()` fija el orden global, mientras que ReAct decide qué herramienta usar dentro de su etapa. La auditoría determinista no usa tokens. La auditoría semántica es **opcional**, para controlar el costo computacional.

In [ ]:
class TechnicalResearchPipeline(dspy.Module):
    def __init__(self, tools, max_iters=3, semantic_check=False):
        super().__init__()
        self.classifier = dspy.Predict(ClassifyQuestion)
        self.researcher = dspy.ReAct(ResearchQuestion, tools=tools, max_iters=max_iters)
        self.writer = dspy.ChainOfThought(WriteAnswer)
        self.semantic_check = semantic_check
        self.checker = dspy.Predict(CheckEvidence) if semantic_check else None

    def forward(self, question: str):
        TOOL_AUDIT.clear()
        RETRIEVED.clear()

        classification = self.classifier(question=question)
        research = self.researcher(
            question=question,
            category=classification.category,
            search_hint=classification.search_hint,
        )
        written = self.writer(
            question=question,
            findings=research.findings,
            source_ids=research.source_ids,
        )
        retrieved_snapshot = dict(RETRIEVED)
        reference_audit = audit_references(written.answer, retrieved_snapshot)
        semantic_result = None
        if self.checker is not None:
            evidence = json.dumps(retrieved_snapshot, ensure_ascii=False)[:11000]
            semantic_result = self.checker(answer=written.answer, evidence=evidence)

        return dspy.Prediction(
            question=question,
            classification=classification,
            research=research,
            written=written,
            answer=written.answer,
            trajectory=getattr(research, "trajectory", None),
            tool_calls=list(TOOL_AUDIT),
            retrieved_documents=retrieved_snapshot,
            reference_audit=reference_audit,
            semantic_audit=semantic_result,
        )

pipeline = TechnicalResearchPipeline(TOOLS, max_iters=MAX_ITERS, semantic_check=False)
print("Pipeline creada")

## 7. Primer caso de estudio

La primera ejecución puede tardar, especialmente si Ollama tiene que cargar el modelo. **Esta celda sí invoca al modelo local**. Si tu modelo no respeta el formato de ReAct, probá con `qwen2.5:7b` o aumentá `max_tokens`.

In [ ]:
CASES = {
    "sqlalchemy": "En SQLAlchemy 2.0 con AsyncSession recibo MissingGreenlet al acceder a una relación ORM. ¿Por qué ocurre y cómo puedo evitarlo?",
    "pydantic": "En Pydantic v2, ¿por qué un campo Optional[str] sin default sigue siendo obligatorio?",
    "fastapi": "¿Por qué FastAPI devuelve HTTP 422 cuando falta un campo requerido en el body?",
    "json": "¿Cómo puedo serializar datetime a JSON en una API FastAPI sin obtener TypeError?",
}

question = CASES["sqlalchemy"]
before = len(lm.history)
started = time.perf_counter()
result = pipeline(question=question)
elapsed = time.perf_counter() - started
calls = lm.history[before:]

print("Pregunta:", question)
print("\nRespuesta:\n", result.answer)
print("\nTiempo (s):", round(elapsed, 2))
print("\nAuditoría de referencias:")
pprint(result.reference_audit)
print("\nLimitaciones:", result.written.caveats)

## 8. Trajectory de ReAct y auditoría de herramientas

La `trajectory` muestra las acciones y observaciones externas del agente. **No es acceso al razonamiento privado del modelo.** El historial de llamadas al modelo contiene los mensajes que DSPy/LiteLLM hayan conservado.

In [ ]:
print("=== ReAct trajectory ===")
pprint(result.trajectory, sort_dicts=False, width=110)

print("\n=== Herramientas invocadas ===")
for i, call in enumerate(result.tool_calls, 1):
    print(f"{i}. {call['tool']}({call['arguments']})")

print("\n=== Fuentes efectivamente recuperadas ===")
for doc_id, doc in result.retrieved_documents.items():
    print(f"{doc_id}: {doc['title']} — {doc['url']}")

## 9. Prompts y respuestas del modelo

`lm.inspect_history()` es la forma más simple de ver los intercambios recientes. El contenido exacto depende de la versión de DSPy y del adaptador LiteLLM.

In [ ]:
print("Llamadas al modelo en esta ejecución:", len(calls))
if calls:
    lm.inspect_history(n=min(len(calls), 8))
else:
    print("No hay historial nuevo; revisá la configuración y el caché.")

In [ ]:
def preview_history(entries, limit=1200):
    for i, entry in enumerate(entries, 1):
        print(f"\n{'='*22} LLAMADA {i} {'='*22}")
        if not isinstance(entry, dict):
            print(str(entry)[:limit]); continue
        print("model:", entry.get("model"))
        print("usage:", entry.get("usage"))
        messages = entry.get("messages") or entry.get("prompt") or []
        if isinstance(messages, list):
            for message in messages:
                if isinstance(message, dict):
                    print("ROLE:", message.get("role"))
                    print(str(message.get("content", ""))[:limit])
                else:
                    print(str(message)[:limit])
        else:
            print(str(messages)[:limit])
        if "response" in entry:
            print("RESPONSE:", str(entry["response"])[:limit])

preview_history(calls, limit=900)

## 10. Tokens y costo

Con Ollama local, **el costo de la API de OpenAI es US$ 0**. Sin embargo, el modelo consume tiempo de CPU/GPU y energía. Algunos adaptadores devuelven métricas de tokens y otros no; si faltan, **no las inventamos**.

El conteo por etapa se puede aproximar por la diferencia de longitud de `lm.history` antes y después de cada etapa. Para atribución exacta, en el próximo experimento instrumentaremos `forward()`.

In [ ]:
def token_usage(entries):
    totals = Counter()
    missing = 0
    for entry in entries:
        usage = entry.get("usage") if isinstance(entry, dict) else None
        if not isinstance(usage, dict):
            missing += 1
            continue
        totals["input_tokens"] += usage.get("prompt_tokens", usage.get("input_tokens", 0)) or 0
        totals["output_tokens"] += usage.get("completion_tokens", usage.get("output_tokens", 0)) or 0
    return {
        "model_calls": len(entries),
        "input_tokens_reported": totals["input_tokens"],
        "output_tokens_reported": totals["output_tokens"],
        "total_tokens_reported": totals["input_tokens"] + totals["output_tokens"],
        "calls_without_usage": missing,
        "openai_api_cost_usd": 0.0,
    }

pprint(token_usage(calls))

## 11. Instrumentación por etapa (prompts/tokens por módulo)

Esta segunda versión guarda el historial de llamadas generado por cada módulo. Así podemos separar el costo computacional de `Predict`, `ReAct`, `ChainOfThought` y del verificador semántico.

In [ ]:
class InstrumentedResearchPipeline(TechnicalResearchPipeline):
    def forward(self, question: str):
        TOOL_AUDIT.clear()
        RETRIEVED.clear()
        stages = {}

        def run_stage(name, fn, **kwargs):
            before = len(lm.history)
            started = time.perf_counter()
            value = fn(**kwargs)
            entries = lm.history[before:]
            stages[name] = {**token_usage(entries), "elapsed_seconds": round(time.perf_counter()-started, 2)}
            return value

        classification = run_stage("classifier", self.classifier, question=question)
        research = run_stage(
            "react", self.researcher, question=question,
            category=classification.category, search_hint=classification.search_hint,
        )
        written = run_stage(
            "writer", self.writer, question=question,
            findings=research.findings, source_ids=research.source_ids,
        )
        retrieved = dict(RETRIEVED)
        ref_audit = audit_references(written.answer, retrieved)
        semantic_result = None
        if self.checker is not None:
            semantic_result = run_stage(
                "semantic_checker", self.checker,
                answer=written.answer,
                evidence=json.dumps(retrieved, ensure_ascii=False)[:11000],
            )
        return dspy.Prediction(
            answer=written.answer, classification=classification,
            research=research, written=written,
            trajectory=getattr(research, "trajectory", None),
            tool_calls=list(TOOL_AUDIT), retrieved_documents=retrieved,
            reference_audit=ref_audit, semantic_audit=semantic_result,
            stage_metrics=stages,
        )

instrumented = InstrumentedResearchPipeline(TOOLS, max_iters=MAX_ITERS, semantic_check=False)
print("Lista para ejecutar una prueba instrumentada")

In [ ]:
# Esta celda hace nuevas llamadas a Ollama.
# Descomentá para comparar métricas por etapa:
# measured = instrumented(question=CASES["pydantic"])
# display(pd.DataFrame.from_dict(measured.stage_metrics, orient="index"))
# pprint(measured.reference_audit)
# print(measured.answer)

## 12. Verificación semántica (opcional)

El verificador adicional lee la respuesta y los documentos efectivamente recuperados. Es una **segunda opinión del LLM**, no una prueba de verdad. Conviene revisar manualmente cada afirmación importante y abrir las URLs oficiales.

In [ ]:
# Esta celda hace nuevas llamadas a Ollama.
# Descomentá para habilitar la segunda evaluación:
# checked_pipeline = InstrumentedResearchPipeline(TOOLS, max_iters=3, semantic_check=True)
# checked = checked_pipeline(question=CASES["fastapi"])
# print(checked.answer)
# pprint(checked.reference_audit)
# pprint(checked.semantic_audit)
# display(pd.DataFrame.from_dict(checked.stage_metrics, orient="index"))

## 13. Casos de prueba y evaluación

Probamos cuatro consultas de distintas bibliotecas. El evaluador determinista solo revisa trazabilidad; para medir **calidad de la respuesta** necesitás una rúbrica manual o un conjunto de respuestas de referencia.

In [ ]:
def evaluate_case(case_id: str, model_pipeline) -> dict:
    question = CASES[case_id]
    before = len(lm.history)
    try:
        pred = model_pipeline(question=question)
        audit = pred.reference_audit
        return {
            "case": case_id,
            "ok": True,
            "category": pred.classification.category,
            "tool_calls": len(pred.tool_calls),
            "cited": len(audit["cited_ids"]),
            "traceable": len(audit["valid_and_retrieved"]),
            "unknown": len(audit["unknown_ids"]),
            "unread": len(audit["known_but_not_retrieved"]),
            "model_calls": len(lm.history) - before,
            "error": "",
        }
    except Exception as exc:
        return {"case": case_id, "ok": False, "error": f"{type(exc).__name__}: {str(exc)[:160]}"}

# Atención: correr todos los casos puede demorar varios minutos.
# rows = [evaluate_case(case_id, instrumented) for case_id in CASES]
# display(pd.DataFrame(rows))

## 14. Experimento A — ¿Qué aporta ReAct frente a una pipeline sin herramientas?

Construimos una baseline que responde con `Predict → ChainOfThought`, sin recuperar documentos. **No es una comparación justa de calidad factual** si una versión tiene evidencia y la otra no; justamente queremos observar ese efecto. Compará además las llamadas al modelo y las referencias trazables.

In [ ]:
class NoToolsBaseline(dspy.Module):
    def __init__(self):
        super().__init__()
        self.classifier = dspy.Predict(ClassifyQuestion)
        self.writer = dspy.ChainOfThought("question, category -> answer")

    def forward(self, question: str):
        category = self.classifier(question=question)
        answer = self.writer(question=question, category=category.category)
        return dspy.Prediction(category=category.category, answer=answer.answer)

baseline = NoToolsBaseline()
# Descomentá para comparar con el mismo caso (hace llamadas locales):
# baseline_result = baseline(question=CASES["sqlalchemy"])
# print("BASELINE:\n", baseline_result.answer)
# print("REACT PIPELINE:\n", result.answer)

## 15. Experimento B — Iteraciones y herramientas

Compará `max_iters=2` frente a `max_iters=5` con la misma pregunta. ReAct puede necesitar varias acciones para buscar, leer y contrastar. Más iteraciones no garantizan mejor calidad, pero suelen incrementar latencia y uso de tokens.

In [ ]:
# Descomentá si querés comparar (cada ejecución usa Ollama):
# comparisons = []
# for max_iters in (2, 5):
#     candidate = InstrumentedResearchPipeline(TOOLS, max_iters=max_iters)
#     pred = candidate(question=CASES["pydantic"])
#     comparisons.append({
#         "max_iters": max_iters,
#         "tools_used": len(pred.tool_calls),
#         "traceable_citations": len(pred.reference_audit["valid_and_retrieved"]),
#         "model_calls": sum(x["model_calls"] for x in pred.stage_metrics.values()),
#     })
# display(pd.DataFrame(comparisons))

## 16. Errores frecuentes y diagnóstico

- **Ollama no responde:** iniciar el servicio y comprobar `http://localhost:11434/api/tags`.
- **Modelo inexistente:** ejecutar `ollama pull qwen2.5:7b` o cambiar `MODEL` por uno instalado.
- **El modelo inventa nombres de herramientas:** usar un modelo con mejor seguimiento de instrucciones, docstrings más claras y menos herramientas.
- **Respuesta sin citas:** mirar `trajectory`, `TOOL_AUDIT` y `reference_audit`; quizá el agente buscó, pero no leyó una ficha.
- **No aparecen tokens:** la integración puede no reportar `usage`; no interpretar cero como ausencia de procesamiento.
- **El modelo se queda sin contexto:** acortar fichas, reducir `max_iters` o aumentar el contexto de Ollama según capacidad del equipo.
- **`fetch_official_page` falla:** las fichas locales siguen funcionando; no asumir que el contenido web fue verificado.
- **Cambios de API de DSPy:** consultar [documentación actual](https://dspy.ai/) y comprobar `dspy.__version__`.

## 17. Ejercicios propuestos

1. Agregar un documento sobre `async_sessionmaker` y comprobar si ReAct lo recupera.
2. Forzar una referencia inventada (`DOC-FAKE-99`) en una respuesta y ejecutar `audit_references`.
3. Cambiar el redactor de `ChainOfThought` a `Predict` y comparar tokens y calidad.
4. Activar `semantic_check=True` y contrastar su juicio con la documentación oficial.
5. Incorporar una quinta etapa que genere un plan de validación de código, sin ejecutar código desconocido.
6. Comparar `llama3.2` y `qwen2.5:7b` con las mismas preguntas y la misma rúbrica.

**Pregunta clave para el análisis:** ¿qué fallos son de *orquestación de la pipeline*, cuáles son de *selección de herramientas por ReAct* y cuáles son de *fidelidad a las evidencias*?

In [ ]:
# Prueba determinista que no llama a ningún modelo:
example = "La carga implícita puede causar MissingGreenlet [DOC-SQLA-01]. Otra afirmación [DOC-FAKE-99]."
pprint(audit_references(example, {"DOC-SQLA-01": DOCS["DOC-SQLA-01"]}))